# P1-DATA-01 — EDA và source profiling của brvehins1

Notebook này dùng script streaming `scripts/profile_brvehins1.py`. Script không sửa raw, đọc CSV theo chunk 100,000 dòng, dùng SQLite tạm ngoài repository để kiểm tra exact duplicate và ghi bằng chứng tại `reports/data/`.

## Kết quả chạy đã ghi nhận

- Tổng số dòng: 1,965,355; năm partition đều có 393,071 dòng và 23 cột cùng schema.
- Exact duplicate logical rows: 14; không deduplicate raw.
- Không có business identifier khách hàng hoặc hợp đồng trong 23 cột.
- Xem artifact đầy đủ tại `reports/data/brvehins1-profile.json`, `reports/data/brvehins1-column-profile.csv` và `reports/data/brvehins1-eda-summary.md`.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROFILE_JSON = PROJECT_ROOT / 'reports' / 'data' / 'brvehins1-profile.json'
PROFILE_SCRIPT = PROJECT_ROOT / 'scripts' / 'profile_brvehins1.py'
assert (PROJECT_ROOT / 'data' / 'raw' / 'brvehins1').is_dir(), 'Không tìm thấy raw canonical.'


In [ ]:
# Chạy lại khi cần tái tạo toàn bộ evidence. Quá trình mất vài phút vì kiểm tra duplicate chính xác.
# subprocess.run([sys.executable, str(PROFILE_SCRIPT), '--chunk-size', '100000'], cwd=PROJECT_ROOT, check=True)

profile = json.loads(PROFILE_JSON.read_text(encoding='utf-8'))
profile['run_metadata']


## Partition, schema và null profile

In [ ]:
partition_frame = pd.DataFrame(profile['partitions'])
null_frame = pd.DataFrame({
    'column': profile['schema']['columns'],
    'missing_count': [profile['missing_counts'][c] for c in profile['schema']['columns']],
    'missing_percent': [profile['missing_percentages'][c] for c in profile['schema']['columns']],
})
display(partition_frame)
display(null_frame.sort_values('missing_count', ascending=False))


## Range, phân bố và metric dẫn xuất

`ClaimFrequency` chỉ có nghĩa khi `ExposTotal > 0`; `LossRatio` chỉ có nghĩa khi `PremTotal > 0`. Các giá trị cực trị được giữ nguyên để business review, không bị cap trong EDA.

In [ ]:
def flatten_profile(section: dict) -> pd.DataFrame:
    rows = []
    for name, values in section.items():
        if not isinstance(values, dict) or 'quantiles' not in values:
            continue
        rows.append({'metric': name, **{k: v for k, v in values.items() if k != 'quantiles'}, **values['quantiles']})
    return pd.DataFrame(rows)

numeric_frame = flatten_profile(profile['numeric_profile'])
derived_frame = flatten_profile(profile['derived_metric_profile'])
display(numeric_frame[['metric', 'valid_count', 'min', 'p50', 'p95', 'p99', 'max', 'negative_count', 'zero_count']])
display(derived_frame[['metric', 'valid_count', 'min', 'p50', 'p95', 'p99', 'max', 'negative_count', 'zero_count']])
profile['derived_metric_profile']['HasClaim']


## Duplicate, state mapping và grain assessment

In [ ]:
print(profile['exact_duplicate_profile'])
print(profile['state_abbreviation_consistency'])
print(profile['cross_field_observations'])
print(profile['grain_assessment'])


## Phân loại quan sát trước khi freeze contract

- VALID: không có số âm; mapping `State` và `StateAb` nhất quán; không có claim amount dương khi tổng claim count bằng 0.
- SUSPICIOUS: 14 dòng logic trùng; đuôi phân bố exposure, premium, claim và loss ratio rất dài.
- INVALID BY CONTRACT: chưa kết luận ở P1-DATA-01 vì contract chưa được đóng băng.
- UNKNOWN / BUSINESS REVIEW REQUIRED: 51,762 dòng có exposure/premium bằng 0; 8 `VehYear` bằng 0; hai trường fire/rob luôn bằng 0; các nullable descriptor phải có policy explicit ở P1-DATA-02.